# Train the vessel detector on Colab (T4 GPU)
1. Runtime → Change runtime type → **T4 GPU**.
2. Upload `maritime_cv.zip` (the project folder **without** `.venv` and `data`) in the Files panel on the left.
3. Run the cells in order. Total time with YOLOv8s: roughly 2–3 hours.

In [ ]:
!nvidia-smi
!unzip -q -o maritime_cv.zip -d /content/maritime_cv
%cd /content/maritime_cv
!pip -q install -r requirements-train.txt

In [ ]:
# Download the Roboflow export (same project/version as in configs/default.yaml)
import os
os.environ['ROBOFLOW_API_KEY'] = input('Roboflow API key: ')
from roboflow import Roboflow
ds = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY']).workspace('naman-kumar-chaudhary') \n       .project('singapore-maritime-bvyeu').version(1).download('yolov8', location='/content/rf_export')

In [ ]:
# Re-split by video (no recording in two splits), then convert to one 'vessel' class
!rm -rf data/maritime
!python scripts/resplit_by_video.py --src /content/rf_export --dst data/maritime
!python scripts/prepare_data.py --root data/maritime --conditions-csv data/maritime/conditions.csv

In [ ]:
# Optional but recommended: keep weights safe if the Colab session disconnects
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/maritime_cv_models

In [ ]:
# Train YOLOv8s (main model). Add yolov8n.pt / yolov8m.pt later for the size comparison.
!python scripts/train.py --data data/maritime/data_vessel.yaml --weights yolov8s.pt --device 0
!cp models/*.pt /content/drive/MyDrive/maritime_cv_models/

In [ ]:
# Table 8: test accuracy overall and per condition (visible / nir / haze) + FPS on the T4
!python scripts/evaluate_detector.py --weights models/vessel_yolov8s_best.pt --data data/maritime/data_vessel.yaml --device 0

In [ ]:
# Table 8 baselines (each takes a while; run when you have time)
!python scripts/train_baselines.py --data data/maritime/data_vessel.yaml --model ssd --epochs 60
!python scripts/train_baselines.py --data data/maritime/data_vessel.yaml --model fasterrcnn --epochs 30

In [ ]:
# Download weights, results and the training curves to your PC
!zip -q -r results_and_models.zip models results runs
!cp results_and_models.zip /content/drive/MyDrive/maritime_cv_models/
from google.colab import files
files.download('results_and_models.zip')